In [43]:
"""
===============================================================================
  PIPELINE DE ENRIQUECIMIENTO MUSICAL — AcousticBrainz
===============================================================================
  Dataset : lastfm_dataset.csv  (59.999 tracks de Last.fm)
  Columnas: name, artist, playcount, listeners, duration,
            mbid, country, genre_tag, rank_global, rank_by_country

  ✅ El dataset ya tiene columna 'mbid' → pipeline directo a AcousticBrainz.

  MBID válidos : 53.294 (88.8 %)
  MBID nulos   :  6.705 (11.2 %) → quedarán con NaN en features de audio

  ⚠️  NOTA SOBRE COBERTURA:
      AcousticBrainz fue discontinuado en noviembre 2022 (solo lectura).
      Cobertura estimada sobre los 53k MBIDs válidos:
        · Low-level  (bpm, key, scale)              : ~40–60 %
        · High-level (danceability, mood, genre_ab) : ~35–55 %
      Los tracks sin datos quedarán con NaN en el DataFrame final.
      Esto es esperable y debe documentarse en el análisis.

  FLUJO:
    1. Cargar dataset → filtrar/samplear
    2. Para cada MBID: GET low-level + GET high-level
    3. Construir df_features con las nuevas columnas
    4. Merge con df original por 'mbid'
    5. Guardar CSV enriquecido

  DEPENDENCIAS:
    pip install requests pandas
===============================================================================
"""


"\n===============================================================================\n  PIPELINE DE ENRIQUECIMIENTO MUSICAL — AcousticBrainz\n===============================================================================\n  Dataset : lastfm_dataset.csv  (59.999 tracks de Last.fm)\n  Columnas: name, artist, playcount, listeners, duration,\n            mbid, country, genre_tag, rank_global, rank_by_country\n\n  ✅ El dataset ya tiene columna 'mbid' → pipeline directo a AcousticBrainz.\n\n  MBID válidos : 53.294 (88.8 %)\n  MBID nulos   :  6.705 (11.2 %) → quedarán con NaN en features de audio\n\n  ⚠️  NOTA SOBRE COBERTURA:\n      AcousticBrainz fue discontinuado en noviembre 2022 (solo lectura).\n      Cobertura estimada sobre los 53k MBIDs válidos:\n        · Low-level  (bpm, key, scale)              : ~40–60 %\n        · High-level (danceability, mood, genre_ab) : ~35–55 %\n      Los tracks sin datos quedarán con NaN en el DataFrame final.\n      Esto es esperable y debe documentarse en 

In [44]:

import os
import time
import logging
from datetime import datetime

import pandas as pd
import requests


In [45]:


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════

# ── Tamaño de muestra ────────────────────────────────────────────────────────
#   None  → procesa los 53k MBIDs válidos (~15–20 horas)
#   2000  → muestra representativa (~45–60 min) ← recomendado para empezar


In [46]:
SAMPLE_SIZE = 2000

# ── Throttling y robustez ────────────────────────────────────────────────────
SLEEP_BETWEEN_CALLS = 0.3   # segundos entre llamadas a la API
MAX_RETRIES         = 3     # reintentos por request fallida
BACKOFF_BASE        = 2     # espera exponencial: 2^intento segundos

# ── Progreso y guardado ──────────────────────────────────────────────────────
LOG_EVERY_N        = 100    # imprimir progreso cada N tracks
CHECKPOINT_EVERY_N = 500    # guardar CSV parcial cada N tracks

# ── Rutas ────────────────────────────────────────────────────────────────────
CSV_INPUT  = "src/lastfm_dataset.csv"
OUTPUT_DIR = "."

# ── Headers para la API (buenas prácticas) ───────────────────────────────────
HEADERS = {
    "User-Agent": "MusicEnrichmentProject/1.0 (student-project@example.com)"
}



In [47]:

# ══════════════════════════════════════════════════════════════════════════════
#  LOGGING
# ══════════════════════════════════════════════════════════════════════════════


In [48]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            os.path.join(OUTPUT_DIR, "pipeline.log"), encoding="utf-8"
        ),
    ],
)
log = logging.getLogger(__name__)


In [49]:


# ══════════════════════════════════════════════════════════════════════════════
#  UTILIDAD: GET con retry + backoff exponencial
# ══════════════════════════════════════════════════════════════════════════════


In [50]:

def _safe_get(url: str) -> requests.Response | None:
    """
    Realiza un GET al endpoint indicado con:
      · Timeout de 10s
      · Retry hasta MAX_RETRIES veces
      · Backoff exponencial en errores 429 / 5xx

    Devuelve:
      · Response object si status 200
      · None si 404 (recurso no existe) o si se agotan los reintentos
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)

            if resp.status_code == 200:
                return resp

            elif resp.status_code == 404:
                # Recurso no existe en AcousticBrainz → sin datos, sin retry
                return None

            elif resp.status_code == 429:
                wait = BACKOFF_BASE ** attempt
                log.warning(f"Rate limit (429). Esperando {wait}s… [intento {attempt}]")
                time.sleep(wait)

            elif resp.status_code >= 500:
                wait = BACKOFF_BASE ** attempt
                log.warning(
                    f"Error de servidor {resp.status_code} en {url}. "
                    f"Esperando {wait}s… [intento {attempt}/{MAX_RETRIES}]"
                )
                time.sleep(wait)

            else:
                log.debug(f"HTTP {resp.status_code} inesperado en {url}")
                return None

        except requests.exceptions.Timeout:
            wait = BACKOFF_BASE ** attempt
            log.warning(f"Timeout. Esperando {wait}s… [intento {attempt}/{MAX_RETRIES}]")
            time.sleep(wait)

        except requests.exceptions.ConnectionError:
            wait = BACKOFF_BASE ** attempt
            log.warning(f"ConnectionError. Esperando {wait}s… [intento {attempt}/{MAX_RETRIES}]")
            time.sleep(wait)

        except Exception as e:
            log.error(f"Error inesperado en GET {url}: {e}")
            return None

    log.debug(f"Agotados {MAX_RETRIES} reintentos para {url}")
    return None


In [51]:


# ══════════════════════════════════════════════════════════════════════════════
#  FUNCIÓN 1: fetch_low_level
# ══════════════════════════════════════════════════════════════════════════════


In [52]:

def fetch_low_level(mbid: str) -> dict:
    """
    Consulta el endpoint LOW-LEVEL de AcousticBrainz.

    URL: https://acousticbrainz.org/{mbid}/low-level

    Features extraídas:
      · bpm   → rhythm.bpm           (float)
      · key   → tonal.key_key        (str, ej: "C", "G#")
      · scale → tonal.key_scale      (str, "major" / "minor")

    Devuelve siempre un dict con las 3 claves.
    Si el MBID no existe o la request falla → valores None.
    """
    result = {"bpm": None, "key": None, "scale": None}

    if not mbid or pd.isna(mbid):
        return result

    url = f"https://acousticbrainz.org/{mbid}/low-level"
    resp = _safe_get(url)
    if resp is None:
        return result

    try:
        data = resp.json()
        result["bpm"]   = data.get("rhythm", {}).get("bpm")
        result["key"]   = data.get("tonal",  {}).get("key_key")
        result["scale"] = data.get("tonal",  {}).get("key_scale")
    except Exception as e:
        log.debug(f"Error parseando low-level (mbid={mbid}): {e}")

    return result


In [53]:


# ══════════════════════════════════════════════════════════════════════════════
#  FUNCIÓN 2: fetch_high_level
# ══════════════════════════════════════════════════════════════════════════════


In [54]:

def fetch_high_level(mbid: str) -> dict:
    """
    Consulta el endpoint HIGH-LEVEL de AcousticBrainz.

    URL: https://acousticbrainz.org/{mbid}/high-level

    Features extraídas:
      · danceability → highlevel.danceability.all.danceable  (float 0–1)
      · mood_happy   → highlevel.mood_happy.all.happy        (float 0–1)
      · genre_ab     → highlevel.genre_dortmund.value        (str: pop, rock…)

    Devuelve siempre un dict con las 3 claves.
    Si el MBID no existe o la request falla → valores None.
    """
    result = {"danceability": None, "mood_happy": None, "genre_ab": None}

    if not mbid or pd.isna(mbid):
        return result

    url = f"https://acousticbrainz.org/{mbid}/high-level"
    resp = _safe_get(url)
    if resp is None:
        return result

    try:
        data = resp.json()
        hl = data.get("highlevel", {})

        result["danceability"] = (
            hl.get("danceability", {}).get("all", {}).get("danceable")
        )
        result["mood_happy"] = (
            hl.get("mood_happy", {}).get("all", {}).get("happy")
        )
        result["genre_ab"] = (
            hl.get("genre_dortmund", {}).get("value")
        )
    except Exception as e:
        log.debug(f"Error parseando high-level (mbid={mbid}): {e}")

    return result


In [55]:


# ══════════════════════════════════════════════════════════════════════════════
#  FUNCIÓN 3: enrich_dataset_with_acousticbrainz
# ══════════════════════════════════════════════════════════════════════════════


In [ ]:

def enrich_dataset_with_acousticbrainz(
    df: pd.DataFrame,
    sample_size: int = None,
) -> pd.DataFrame:
    """
    Pipeline completo de enriquecimiento con AcousticBrainz.

    Parámetros
    ----------
    df          : DataFrame con columna 'mbid' (el dataset de Last.fm)
    sample_size : Número de tracks a enriquecer.
                  None → todos los tracks con MBID válido.

    Proceso
    -------
    1. Filtra tracks con NAME válido
    2. Aplica sampling si sample_size no es None
    3. Itera: low-level + high-level por cada MBID
    4. Construye df_features (bpm, key, scale, danceability, mood_happy, genre_ab)
    5. Merge con df original por 'mbid'
    6. Guarda checkpoints parciales y resultado final

    Devuelve
    --------
    df_enriched : DataFrame con columnas originales + 6 nuevas features de audio
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M")

    # ── Separar tracks con y sin name ────────────────────────────────────────
    df_valid   = df.dropna(subset=["mbid"]).copy()
    df_no_name = df[df["name"].isna()].copy()

    log.info("=" * 62)
    log.info("  ENRIQUECIMIENTO CON ACOUSTICBRAINZ")
    log.info("=" * 62)
    log.info(f"  Total tracks          : {len(df):,}")
    log.info(f"  Con MBID válido       : {len(df_valid):,}")
    log.info(f"  Sin MBID (→ NaN)      : {len(df_no_mbid):,}")

    # ── Aplicar muestra si se indica ─────────────────────────────────────────
    if sample_size is not None:
        n = min(sample_size, len(df_valid))
        df_sample = df_valid.sample(n=n, random_state=42).reset_index(drop=True)
        log.info(f"  Muestra seleccionada  : {len(df_sample):,} (random_state=42)")
    else:
        df_sample = df_valid.reset_index(drop=True)
        log.info(f"  Procesando TODOS los tracks con MBID")
    log.info("=" * 62)

    # ── Iteración principal ───────────────────────────────────────────────────
    results    = []
    ok_ll      = 0
    ok_hl      = 0
    fail_ll    = 0
    fail_hl    = 0
    total      = len(df_sample)
    start_time = time.time()

    for i, row in enumerate(df_sample.itertuples(index=False), start=1):
        mbid = row.mbid

        # Consulta Low-Level
        ll = fetch_low_level(mbid)
        time.sleep(SLEEP_BETWEEN_CALLS)

        # Consulta High-Level
        hl = fetch_high_level(mbid)
        time.sleep(SLEEP_BETWEEN_CALLS)

        # Contabilizar éxitos / fallos
        if ll["bpm"] is not None:
            ok_ll += 1
        else:
            fail_ll += 1

        if hl["genre_ab"] is not None:
            ok_hl += 1
        else:
            fail_hl += 1

        # Guardar resultado de este track
        results.append({"mbid": mbid, **ll, **hl})

        # ── Progreso ──────────────────────────────────────────────────────────
        if i % LOG_EVERY_N == 0 or i == total:
            elapsed   = time.time() - start_time
            req_per_s = (i * 2) / elapsed if elapsed > 0 else 0
            eta_min   = ((total - i) * 2) / (req_per_s * 60) if req_per_s > 0 else 0
            log.info(
                f"[{i:>5}/{total}] "
                f"LL ✓{ok_ll} ✗{fail_ll} ({ok_ll/i*100:.0f}%) | "
                f"HL ✓{ok_hl} ✗{fail_hl} ({ok_hl/i*100:.0f}%) | "
                f"{req_per_s:.1f} req/s | ETA {eta_min:.1f} min"
            )

        # ── Checkpoint parcial ────────────────────────────────────────────────
        if i % CHECKPOINT_EVERY_N == 0:
            ckpt_df   = pd.DataFrame(results)
            ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint_ab_{i}_{ts}.csv")
            ckpt_df.to_csv(ckpt_path, index=False)
            log.info(f"  💾 Checkpoint guardado: {ckpt_path}")

    # ── Construir DataFrame de features ──────────────────────────────────────
    df_features = pd.DataFrame(results)
    df_features = df_features.drop_duplicates(subset="mbid")

    # ── Merge con la muestra original ─────────────────────────────────────────
    df_enriched = df_sample.merge(df_features, on="mbid", how="left")

    # ── Guardar resultado final ───────────────────────────────────────────────
    output_path = os.path.join(OUTPUT_DIR, f"lastfm_enriched_{ts}.csv")
    df_enriched.to_csv(output_path, index=False)

    # ── Resumen final ─────────────────────────────────────────────────────────
    log.info("\n" + "=" * 62)
    log.info("  RESUMEN FINAL")
    log.info("=" * 62)
    log.info(f"  Tracks procesados     : {total:,}")
    log.info(f"  Low-level OK          : {ok_ll:,} ({ok_ll/total*100:.1f}%)")
    log.info(f"  High-level OK         : {ok_hl:,} ({ok_hl/total*100:.1f}%)")
    log.info(f"  Low-level sin datos   : {fail_ll:,} ({fail_ll/total*100:.1f}%)")
    log.info(f"  High-level sin datos  : {fail_hl:,} ({fail_hl/total*100:.1f}%)")
    log.info(f"  Archivo guardado en   : {output_path}")
    log.info("=" * 62)

    return df_enriched


In [57]:


# ══════════════════════════════════════════════════════════════════════════════
#  EJECUCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════


In [58]:
CSV_INPUT  = "src/lastfm_dataset.csv"
if __name__ == "__main__":

    # ── 1. Cargar dataset ─────────────────────────────────────────────────────
    log.info(f"Cargando dataset: {CSV_INPUT}")
    df_clean = pd.read_csv(CSV_INPUT)
    log.info(f"Shape: {df_clean.shape} | MBIDs válidos: {df_clean['mbid'].notna().sum():,}")

    # ── 2. Elegir modo de ejecución ───────────────────────────────────────────

    # OPCIÓN A — Muestra de 2.000 tracks (~45–60 min) ← recomendado para empezar
    df_enriched = enrich_dataset_with_acousticbrainz(df_clean, sample_size=2000)

    # OPCIÓN B — Muestra de 5.000 tracks (~2–3 horas)
    # df_enriched = enrich_dataset_with_acousticbrainz(df_clean, sample_size=5000)

    # OPCIÓN C — Dataset completo, todos los MBIDs válidos (~15–20 horas)
    # df_enriched = enrich_dataset_with_acousticbrainz(df_clean, sample_size=None)

    # ── 3. Inspección rápida del resultado ───────────────────────────────────
    cols_show = ["name", "artist", "genre_tag", "country",
                 "bpm", "key", "scale", "danceability", "mood_happy", "genre_ab"]

    print("\n── Primeras filas del dataset enriquecido ──")
    print(df_enriched[cols_show].head(10).to_string(index=False))

    print("\n── Cobertura de features de audio ──")
    feature_cols = ["bpm", "key", "scale", "danceability", "mood_happy", "genre_ab"]
    for col in feature_cols:
        n   = df_enriched[col].notna().sum()
        pct = n / len(df_enriched) * 100
        bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {col:<15} {bar}  {n:>5} / {len(df_enriched):,}  ({pct:.1f}%)")

    print("\n── Estadísticas básicas de features numéricas ──")
    print(df_enriched[["bpm", "danceability", "mood_happy"]].describe().round(3))


10:45:15  INFO      Cargando dataset: src/lastfm_dataset.csv
10:45:15  INFO      Shape: (59999, 10) | MBIDs válidos: 53,294
10:45:15  INFO      ==============================================================
10:45:15  INFO        ENRIQUECIMIENTO CON ACOUSTICBRAINZ
10:45:15  INFO      ==============================================================
10:45:15  INFO        Total tracks          : 59,999
10:45:15  INFO        Con MBID válido       : 53,294
10:45:15  INFO        Sin MBID (→ NaN)      : 6,705
10:45:15  INFO        Muestra seleccionada  : 2,000 (random_state=42)
10:45:15  INFO      ==============================================================
10:46:37  INFO      [  100/2000] LL ✓8 ✗92 (8%) | HL ✓8 ✗92 (8%) | 2.4 req/s | ETA 26.0 min
10:48:01  INFO      [  200/2000] LL ✓13 ✗187 (6%) | HL ✓13 ✗187 (6%) | 2.4 req/s | ETA 24.9 min
10:49:22  INFO      [  300/2000] LL ✓16 ✗284 (5%) | HL ✓16 ✗284 (5%) | 2.4 req/s | ETA 23.4 min


KeyboardInterrupt: 

In [60]:
df_enriched = enrich_dataset_with_acousticbrainz(df_clean, sample_size=10)

10:52:05  INFO      ==============================================================
10:52:05  INFO        ENRIQUECIMIENTO CON ACOUSTICBRAINZ
10:52:05  INFO      ==============================================================
10:52:05  INFO        Total tracks          : 59,999
10:52:05  INFO        Con MBID válido       : 53,294
10:52:05  INFO        Sin MBID (→ NaN)      : 6,705
10:52:05  INFO        Muestra seleccionada  : 10 (random_state=42)
10:52:05  INFO      ==============================================================


10:52:13  INFO      [   10/10] LL ✓0 ✗10 (0%) | HL ✓0 ✗10 (0%) | 2.5 req/s | ETA 0.0 min
10:52:13  INFO      
10:52:13  INFO        RESUMEN FINAL
10:52:13  INFO      ==============================================================
10:52:13  INFO        Tracks procesados     : 10
10:52:13  INFO        Low-level OK          : 0 (0.0%)
10:52:13  INFO        High-level OK         : 0 (0.0%)
10:52:13  INFO        Low-level sin datos   : 10 (100.0%)
10:52:13  INFO        High-level sin datos  : 10 (100.0%)
10:52:13  INFO        Archivo guardado en   : ./lastfm_enriched_20260327_1052.csv
10:52:13  INFO      ==============================================================


In [61]:
df_enriched

,name,artist,playcount,listeners,duration,mbid,country,genre_tag,rank_global,rank_by_country,bpm,key,scale,danceability,mood_happy,genre_ab
0,This Could Be Love,Alkaline Trio,0,0,227,042c0c6b-0e01-423f-adb2-14bf0948e3d4,UNKNOWN,alternative,NaN,NaN,None,None,None,None,None,None
1,1999,Charli xcx,0,0,210,0379ed24-fd11-4dd2-b884-c21a14f22238,UNKNOWN,pop,NaN,NaN,None,None,None,None,None,None
2,Do What You Gotta Do,Nina Simone,0,0,214,05f863d9-0c54-4d9e-b7e2-9e0d1241dad9,UNKNOWN,jazz,NaN,NaN,None,None,None,None,None,None
3,The Meaning of Life,The Offspring,0,0,176,06c6adce-6596-4798-831a-01fd84bc38b1,UNKNOWN,punk,NaN,NaN,None,None,None,None,None,None
4,365,Charli xcx,0,86,208,296fbbd4-57b1-41f6-8394-da2ea3bb9c10,Japan,UNKNOWN,NaN,1464.0,None,None,None,None,None,None
5,Callas Went Away,Enigma,0,0,264,035f56c4-77c4-306a-9940-da2813965bb2,UNKNOWN,ambient,NaN,NaN,None,None,None,None,None,None
6,What You Know,Two Door Cinema Club,0,1202,190,02c47925-5759-4d3f-9055-cb9ef45e5a01,Germany,UNKNOWN,NaN,606.0,None,None,None,None,None,None
7,Other People,Beach House,0,0,265,21385435-283a-3296-b8bb-27aca961f51f,UNKNOWN,female vocalists,NaN,NaN,None,None,None,None,None,None
8,American Boy,Estelle,0,0,203,03c32a88-5cb3-4bad-854c-8f64784442f3,UNKNOWN,female vocalists,NaN,NaN,None,None,None,None,None,None
9,Songbird,Kenny G,0,0,305,001d4980-8678-368d-9b30-02b36ddf89b9,UNKNOWN,instrumental,NaN,NaN,None,None,None,None,None,None


In [ ]:
bpm